# 01 — Fit a sphere (Student workbook)

**Time:** 65 minutes
**Question:** what can a SANS curve tell us about a dilute spherical particle?

You will connect the analytical sphere amplitude to a SasModels fit, choose
bounds, and diagnose the result using normalized residuals.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
values = np.genfromtxt(DATA / 'sphere_reference.csv', delimiter=',', names=True)
q, intensity, uncertainty = values['Q'], values['I'], values['dI']


## Exercise 1 — Reduce the model

For a homogeneous sphere,

$$
F(q)=3\frac{\sin(qr)-qr\cos(qr)}{(qr)^3}, \qquad
I(q)=\mathrm{scale}\,V(\Delta\rho)^2F(q)^2+\mathrm{background}.
$$

Explain why volume is not an independent fifth shape parameter, then complete
the amplitude function. Ignore absolute scaling in this first visual test.

In [ ]:
def sphere_amplitude(q, radius):
    qr = q * radius
    # TODO: implement 3 * (sin(qr) - qr*cos(qr)) / (qr**3)
    return np.ones_like(qr)


trial_radius = 50.0  # TODO: change this and rerun the plot
trial_shape = sphere_amplitude(q, trial_radius) ** 2


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(q, intensity / intensity.max(), 'o', ms=3, label='normalized data')
ax.loglog(q, trial_shape / trial_shape.max(), label=f'trial r = {trial_radius:g} Å')
ax.set(xlabel=r'$q$ / Å$^{-1}$', ylabel='normalized intensity')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Exercise 2 — The fitting workflow

Every fit follows the same sequence:

1. create `SANSFitter()`;
2. load a reduced dataset with `load_data(path)`;
3. select a SasModels model with `set_model('model_name')`;
4. inspect its parameters with `get_params()`;
5. configure each parameter with `set_param(...)`;
6. call `fit(...)` and inspect the residuals.

The first two steps are demonstrated below. You will choose the model.

In [ ]:
fitter = SANSFitter()
fitter.load_data(str(DATA / 'sphere_reference.csv'))


In [ ]:
# `set_model` takes a SasModels model name as a string.
# TODO: replace None with the model name for a homogeneous sphere.
model_name = None
if model_name is None:
    raise NotImplementedError('Choose a SasModels model')

fitter.set_model(model_name)

# `get_params` lists every parameter supplied by that model.
fitter.get_params()


### Changing parameters

The general form is:

```python
fitter.set_param(
    'parameter_name',
    value=initial_guess,
    min=lower_bound,
    max=upper_bound,
    vary=True,
)
```

- `value` is the optimizer's starting point, not the answer.
- `min` and `max` express physically defensible limits.
- `vary=True` lets the optimizer change the parameter.
- `vary=False` fixes independently known quantities such as SLD.

Keep both SLDs fixed. Choose and justify initial values and bounds for radius,
scale, and background. Afterward, call `get_params()` to audit your setup.

In [ ]:
# One fixed parameter is demonstrated below. `vary=False` tells the
# optimizer to retain this independently known value.
fitter.set_param('sld', value=4.2, vary=False)

# YOUR CODE HERE
# 1. Fix `sld_solvent` at 1.0.
# 2. Configure radius, scale, and background with an initial value and bounds.
# 3. Set `vary=True` for those three fitted parameters.
raise NotImplementedError('Configure the remaining model parameters')

fitter.get_params()


### Running the optimizer

`fit(engine='lmfit', method='least_squares')` performs a bounded local
least-squares fit. The returned dictionary contains the objective value and
fitted parameters. `plot_results(...)` overlays the model and displays
normalized residuals.

In [ ]:
# `fit` accepts an optimization engine and an engine-specific method.
# For a bounded local fit, use engine='lmfit' and method='least_squares'.
# Store the returned dictionary in `result`, then call `plot_results`.

# YOUR CODE HERE
raise NotImplementedError('Fit the configured model and plot its residuals')


## Exercise 3 — Criticize, do not merely accept

Record:

- fitted radius and uncertainty;
- whether residuals look randomly distributed around zero;
- any fitted parameter touching a bound;
- one experimental systematic that this model ignores.

In [ ]:
# YOUR CODE HERE
# Extract the fitted radius, scale, and background from `result`.
# Then write your scientific interpretation in a new Markdown cell.
raise NotImplementedError('Summarize and interpret the fit')


## Extension — Bad starting values

Change the initial radius to 15 Å while retaining broad bounds. Does the local
optimizer reach the same answer? If not, try BUMPS differential evolution with
`fitter.fit(engine='bumps', method='de', steps=200)`. What does this reveal
about optimization versus inference?